# 17 · Vision – the Avengers assemble 🧪

Vision is not one mind: it was built from J.A.R.V.I.S., the Mind Stone and Tony's and Bruce's work. Our finale connects **independent agents** into one system with **A2A** (*Agent2Agent*), the open protocol for agents to discover and call each other.

```
you ─▶ jarvis-orchestrator ──(A2A tool)──▶ friday-finance  (own endpoint, own instructions)
              └──(MCP tool)──▶ stark-mcp (workshop facts)
```

| Part | What it is |
|---|---|
| **Agent card** | the discovery document of an A2A agent (description, skills) |
| **Incoming A2A** | switch on the A2A endpoint of a Foundry prompt agent (`update_details`) |
| **A2A connection** | project connection that stores the remote agent's URL and auth |
| **A2A tool** | `A2ATool` on the calling agent |

> 🧪 **Preview-adjacent.** A2A v1.0 is GA in Foundry; the Python SDK surface (`update_details`, `A2ATool`) needs `azure-ai-projects>=2.5.0`. Limits: text only, no streaming, JSON-RPC only. Calls need Entra ID and the **Foundry Agent Consumer** role. Deep dive: [docs/capstone.md](../docs/capstone.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_a2a import *   # shows A2A replies and orchestrator turns as Markdown: plumbing only

import json, requests
from azure.ai.projects.models import (A2AProtocolConfiguration, A2AProtocolVersion, A2ATool, AgentCard, AgentCardSkill,
                                      AgentEndpointConfig, MCPTool, ProtocolConfiguration, PromptAgentDefinition,
                                      ResponsesProtocolConfiguration)
project_openai = openai_client()
agents_to_delete = []          # agent names, deleted in 🧹 Clean up
conversations_to_delete = []   # conversation ids, deleted in 🧹 Clean up

## 1 · The remote specialist: `friday-finance`
A normal prompt agent. Its instructions embed the **invoice-review skill** from chapter 15 as plain text (direct injection, chapter 07), and it can use our MCP server's `check_invoice_approval` tool. A prompt agent has no shell, so the skill's scripts don't run: F.R.I.D.A.Y. follows the skill's fallback, runs checks 1 to 3 and 7 herself and says that 4 to 6 were not checked.

In [ ]:
friday_agent = project().agents.create_version(
    agent_name="friday-finance",
    definition=PromptAgentDefinition(
        model=model("chat"),
        instructions="You are F.R.I.D.A.Y., the finance specialist. Review invoices strictly with this checklist:\n" + skill_body("invoice-review"),
        tools=[MCPTool(server_label="stark", server_url=cfg("STARK_MCP_URL"), project_connection_id="stark-mcp",
                       require_approval="never", allowed_tools=["check_invoice_approval"])],
    ),
)
agents_to_delete.append("friday-finance")
print("created", friday_agent.name, "v", friday_agent.version)

## 2 · Expose it over A2A
One call sets the **agent card** and turns on the A2A protocol next to Responses.

In [ ]:
project().agents.update_details(
    agent_name="friday-finance",
    agent_endpoint=AgentEndpointConfig(protocol_configuration=ProtocolConfiguration(
        responses=ResponsesProtocolConfiguration(), a2a=A2AProtocolConfiguration())),
    agent_card=AgentCard(version="1.0", description="Reviews supplier invoices and checks approval limits.",
        skills=[AgentCardSkill(id="invoice-review", name="Invoice review", description="Reviews invoice data against the checklist and approval limits.")]),
)
FRIDAY_A2A_URL = f"{cfg('STARK_PROJECT_ENDPOINT')}/agents/friday-finance/endpoint/protocols/a2a"   # base URL of her A2A endpoint
print(FRIDAY_A2A_URL)

## 3 · Discover it like any A2A client
Plain HTTP with an Entra token (scope `https://ai.azure.com/.default`). Anonymous access is not supported. Version **v1.0** card.

**Goal:** read F.R.I.D.A.Y.'s agent card the way any A2A client would, with your own token.

**Expected:** `description` and `skills` are the ones set in section 2: "Reviews supplier invoices and checks approval limits." and one skill, `invoice-review`. ❓ `name` and `protocolVersion` are filled in by Foundry and were not checked.

In [ ]:
user_token = credential().get_token("https://ai.azure.com/.default").token   # your az login identity
friday_card = requests.get(f"{FRIDAY_A2A_URL}/agentCard/v1.0", headers={"Authorization": f"Bearer {user_token}"}).json()
print(json.dumps({k: friday_card.get(k) for k in ("name", "description", "protocolVersion", "skills")}, indent=2))

### Call it with the open-source A2A SDK
The `a2a-sdk` reads `protocolVersion` from the card and negotiates v1.0. This is exactly what a non-Foundry agent (LangGraph, your own service) would do. `get_stream_response_text()` from the same SDK takes the text out of each reply, whether it is a `message` or a `task`. `show_a2a_replies()` ([stark_a2a.py](stark_a2a.py)) shows the question, the joined text and the payload kinds.

**Goal:** a client outside Foundry sends F.R.I.D.A.Y. one message over A2A.

| Input | Value |
|---|---|
| question | `friday_question`: invoice 9930 from Oscorp Materials, 99,792 USD, approver Pepper Potts |
| identity | your token (`user_token`), fetched again here because it expires |

**Expected:** an answer from F.R.I.D.A.Y. She calls `check_invoice_approval`, and 99,792 USD is within Pepper Potts's limit of 250,000. ❓ Her verdict: the question gives no invoice date, and the checklist never approves an invoice with a missing required field, so `NEEDS HUMAN REVIEW` or `REJECT` is as likely as `APPROVE`.

In [ ]:
import httpx
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.helpers import new_text_message
from a2a.types.a2a_pb2 import Role, SendMessageRequest

friday_question = "Invoice 9930 from Oscorp Materials totals 99,792 USD. Can Pepper Potts approve it?"
user_token = credential().get_token("https://ai.azure.com/.default").token

async with httpx.AsyncClient(headers={"Authorization": f"Bearer {user_token}"}, timeout=httpx.Timeout(120.0)) as http:
    agent_card = await A2ACardResolver(httpx_client=http, base_url=FRIDAY_A2A_URL, agent_card_path="agentCard/v1.0").get_agent_card()
    a2a_client = await create_client(agent=agent_card, client_config=ClientConfig(streaming=False, httpx_client=http))
    request = SendMessageRequest(message=new_text_message(friday_question, role=Role.ROLE_USER))
    friday_replies = [response async for response in a2a_client.send_message(request)]
    await a2a_client.close()

show_a2a_replies("friday-finance", friday_question, friday_replies)

## 4 · Let another Foundry agent call it
Step 1: a **connection** to the target (ARM). Step 2: an agent with the **A2A tool**. The caller's identity (agent identity) needs the *Foundry Agent Consumer* role on the target.

In [ ]:
PROJECT_PATH = f"{account_path()}/projects/{cfg('STARK_FOUNDRY_PROJECT')}"
CONNECTION_PATH = f"{PROJECT_PATH}/connections/friday-a2a"
CONNECTION_API = "2025-04-01-preview"
connection_body = {"properties": {"authType": "AgenticIdentityToken", "category": "RemoteA2A", "target": FRIDAY_A2A_URL,
                                  "audience": "https://ai.azure.com", "Credentials": {}, "metadata": {}}}
connection_response = arm("PUT", CONNECTION_PATH, CONNECTION_API, connection_body)
print(connection_response.status_code, "" if connection_response.ok else connection_response.text)

In [ ]:
friday_connection = project().connections.get("friday-a2a")
orchestrator_agent = project().agents.create_version(
    agent_name="jarvis-orchestrator",
    definition=PromptAgentDefinition(
        model=model("chat"),
        instructions=("You are J.A.R.V.I.S., the orchestrator. For anything about invoices or approvals, delegate to F.R.I.D.A.Y. "
                      "with the A2A tool and relay her answer. For suits and workshop facts use the stark tools. Be brief."),
        tools=[A2ATool(a2a_version=A2AProtocolVersion.V1_0, project_connection_id=friday_connection.id),
               MCPTool(server_label="stark", server_url=cfg("STARK_MCP_URL"), project_connection_id="stark-mcp",
                       require_approval="never", allowed_tools=["list_suits", "get_suit", "get_workshop_status"])],
    ),
)
agents_to_delete.append("jarvis-orchestrator")
print("created", orchestrator_agent.name, "v", orchestrator_agent.version)

In [ ]:
# Grant the orchestrator's own identity access to the target: the Foundry Agent Consumer role on the project.
import uuid
AGENT_CONSUMER = "eed3b665-ab3a-47b6-8f48-c9382fb1dad6"   # built-in role definition id
try:
    orchestrator_principal = project().agents.get(agent_name="jarvis-orchestrator").instance_identity.principal_id
    role_definition_id = f"/subscriptions/{cfg('AZURE_SUBSCRIPTION_ID')}/providers/Microsoft.Authorization/roleDefinitions/{AGENT_CONSUMER}"
    assignment_name = uuid.uuid5(uuid.NAMESPACE_URL, PROJECT_PATH + orchestrator_principal + AGENT_CONSUMER)   # same name on a re-run, no duplicates
    role_response = arm("PUT", f"{PROJECT_PATH}/providers/Microsoft.Authorization/roleAssignments/{assignment_name}", "2022-04-01",
                        {"properties": {"principalId": orchestrator_principal, "principalType": "ServicePrincipal",
                                        "roleDefinitionId": role_definition_id}})
    print("role assignment:", role_response.status_code,
          "(already assigned)" if role_response.status_code == 409 else ("" if role_response.ok else role_response.text))
except Exception as exc:
    print("Assign 'Foundry Agent Consumer' to the orchestrator's identity manually:", type(exc).__name__)

## 5 · The whole system in one question
Role assignments need a minute to propagate. `ask_orchestrator()` ([stark_a2a.py](stark_a2a.py)) sends the question with `ask_agent()` and retries up to three times, 45 seconds apart, while the call fails (a 403 until the role is there). Then it shows the prompt, the whole answer and a check table: the tool calls in the response (`tool_calls()` from [stark_helpers.py](stark_helpers.py)), whether one is an MCP call, whether an item type names A2A, and the last verdict keyword of the final message.

**Goal:** one question that needs both tools: a workshop fact (MCP) and an invoice decision (A2A to F.R.I.D.A.Y.).

| Input | Value |
|---|---|
| question | `system_question`: the arc reactor, and invoice 9930 for 99,792 USD with approver Pepper Potts |
| conversation | `capstone_conversation`, new in this cell |

**Expected:** the arc reactor is online (3.0 GW in the static `stark-mcp` data), from an MCP call to `get_workshop_status`. For the invoice, the orchestrator delegates to F.R.I.D.A.Y. with the A2A tool and relays her answer: within Pepper's limit, and ❓ the same open verdict as in section 3, since the question names neither vendor nor date. ❓ The output item type of an A2A tool call was not checked, so the check table lists every tool call it finds.

In [ ]:
system_question = "Is the arc reactor online, and can Pepper Potts approve invoice 9930 for 99,792 USD?"
capstone_conversation = project_openai.conversations.create()
conversations_to_delete.append(capstone_conversation.id)
ask_orchestrator("jarvis-orchestrator", system_question, capstone_conversation.id)

Open **Traces** (chapter 03) and the conversation: you see the orchestrator's A2A call and, as a separate agent run, Friday's own work with her MCP call – two agents, two identities, one trace story.

## 6 · Prove it (chapter 10 again)
Run the quality gate on the orchestrator before you trust it.

**Goal:** the release check. The orchestrator answers the four queries in [capstone-queries.jsonl](../data/capstone-queries.jsonl), a judge model scores `task_adherence`, and the gate needs a pass rate of at least 0.6.

| Query | Needs |
|---|---|
| Which Iron Man suits are still operational? | `stark-mcp` (none is: museum, destroyed, destroyed, retired) |
| Can Pepper Potts approve an invoice of 99,792 USD? | F.R.I.D.A.Y. over A2A (limit 250,000) |
| Is the arc reactor online? | `stark-mcp` (online) |
| Can Tony Stark approve an invoice of 3,332 USD? | F.R.I.D.A.Y. over A2A (limit 1,000,000) |

**Expected:** ❓ `quality gate passed (>= 0.6)`: with four queries that takes three passes, because two of four is 0.5. The judge decides, so the result can differ between runs. When it fails, the cell prints `GATE FAILED` instead of raising, so the clean-up still runs. The score table below it shows which query failed and why (`eval_scores()` shortens the reasons).

In [ ]:
capstone_dataset = project().datasets.upload_file(name="stark-capstone", version="1", file_path=str(ROOT / "data" / "capstone-queries.jsonl"))
capstone_eval = project_openai.evals.create(name="capstone", data_source_config=item_schema({"query": {"type": "string"}}, required=["query"]),
    testing_criteria=[criterion("task_adherence", "task_adherence", {"query": "{{item.query}}", "response": "{{sample.output_items}}"})])
eval_run = project_openai.evals.runs.create(eval_id=capstone_eval.id, name="orchestrator",
                                            data_source=agent_target(capstone_dataset.id, "jarvis-orchestrator"))
eval_run = wait_eval_run(capstone_eval.id, eval_run.id, project_openai)
try:
    assert_pass_rate(eval_run, minimum=0.6)
except AssertionError as exc:
    print("GATE FAILED:", exc)
display(eval_scores(capstone_eval.id, eval_run.id, project_openai))

## 👀 In the portal
**Operate → Assets** 🧪 and **Build → Agents**: `jarvis-orchestrator` and `friday-finance`; **Evaluation**: `capstone`. Compare their traces.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up

In [ ]:
for conversation_id in conversations_to_delete:
    project_openai.conversations.delete(conversation_id)
conversations_to_delete.clear()
try: project_openai.evals.delete(capstone_eval.id)   # skipped when section 6 did not run
except Exception: pass
for name in agents_to_delete:
    delete_agent(name)
print(arm("DELETE", CONNECTION_PATH, CONNECTION_API).status_code, "(connection friday-a2a)")
delete_dataset("stark-capstone")

Remove the **whole workshop** from Azure (models, monitoring, function): `azd down --purge`.

## 🎯 Recap: the moral
You started with a robot arm that could answer one question. Now you have models, memory, tools, skills, code-hosted agents, a harness that runs the loop for you, evidence, senses, guardrails, workflows and agents that talk to each other. **Foundry is the workshop; you decide what to build.**

> 💡 Next: your own suit. [story.md](../story.md) links every chapter, if you want to revisit one.